# Preliminary Session 5: A SLAM demonstration and introduction to Active Sensing

## Redes de Sensores y Sistemas Autónomos 
### Grado en Ingeniería de las Tecnologías de Telecomunicación
### Universidad de Sevilla

David Alejo Teissière

## Introduction

In this practical session we will make our Turtlebot to be able to map an unknown area. To this end, we will need a SLAM algorithm (Simultaneous Localization and Mapping).

To sum up, in this lesson we will learn to:

* Set up a turtlebot3 gazebo simulation
* Use the NAV2 autonomous navigation stack so that the Turtlebot3 can fulfill basic navigation goal commands
* Configure a SLAM algorithm to generate in real-time a map of the environment
* Use the m-explore package to autonomously generate a map using frontier-based exploration.

Let's go!

## Introduction to active sensing

The main goal of this session is to autonomously generate a map of the environment. This map will be stored as an Occupancy Grid (see Fig. 1). 

As we want this exploration to be autonomous. Hence, we will use a frontier based exploration algorithm so that the Turtlebot will be assigned with a waypoint in the closest frontier as the new Navigation Goal. The map will be generated by using the cartographer SLAM algorithm.


<figure style="text-align:center">
  <img src="images/inflated_maze.png" alt="" width=700>
  <figcaption>Fig. 1: Geometry map of the Turlebot3 Maze environment.  </figcaption>
</figure>


In [ ]:
###### Example of map metadata ##########
image: testmap.png
resolution: 0.1
origin: [0.0, 0.0, 0.0]
occupied_thresh: 0.65
free_thresh: 0.196
negate: 0

#### Simultaneous localization and mapping (SLAM)

We will run a Simultaneous Localization and Mapping algorithm to let her acquire knowledge of the environment thanks to the information gathered by her onboard 2D LiDAR sensor. To this end, we can use the `cartographer` package available on ROS 2. The SLAM node within this package generates the map using a Pose-Graph SLAM algorithm. The map is stored in a grid similar to the one used in the map server package.

Figure 1 shows the main block diagram of the Cartographer algorithm. There, you can distinguish between the front-end part (the upper half) and the back-end, which optimizes the graph whenever new loop-closing constraints are added. This back-end is performed in a separate thread, as it may be time-consuming.

<figure style="text-align:center">
  <img src="images/high_level_system_overview.webp" alt="" width=700>
  <figcaption>Fig. 2: Main block diagram of the Cartographer algorithm.  </figcaption>
</figure>

More information: Please refer to the [paper](https://static.googleusercontent.com/media/research.google.com/en//pubs/archive/45466.pdf).
                  Also refer to (https://google-cartographer.readthedocs.io/en/latest/).


#### Launching it all

The basic algorithms for SLAM and exploration are implemented in basic nodes of ROS. We use the following packages:

* cartographer: Implements a two Step Local-Global SLAM with a Scan Matching algorithm and a Pose Graph optimization algorithms, respectively (https://google-cartographer-ros.readthedocs.io/en/latest/index.html)
* nav2: Implements the navigation stack for ROS2, which has some options available for global and local planning.
* explore_lite: You have to download the source code from [here](https://github.com/robo-friends/m-explore-ros2) into your ROS2 workspace and compile it (catkin_make)

Once you have downloaded and compiled the exploration code. You can launch the autonomous exploration suite by using:

`(rssa_docker) > ros2 launch nav2_bringup tb3_simulation_launch.py slam:=True `

__Exercise 1__ Execute the SLAM simulation the above program and try to generate a map of the environment. You can do it by manually selecting goals of unexplored areas with the Goal Pose tool of RViz2.

__Exercise 2__ Repeat the experiment in different environments. For example turtlebot3_house.

```
(rssa_docker) > export GZ_SIM_RESOURCE_PATH=$GZ_SIM_RESOURCE_PATH:/opt/ros/jazzy/share/turtlebot3_gazebo/models
(rssa_docker) > ros2 launch nav2_bringup tb3_simulation_launch.py slam:=True world:=/opt/ros/jazzy/share/turtlebot3_gazebo/worlds/turtlebot3_house.world 

```

##### Saving the map

At the end of the experiment, we can save the map progress by using the `map_saver_cli` tool of the `nav2_map_server` package.

`(rssa_docker)> ros2 run nav2_map_server map_saver_cli`

### Introduction to active perception: Frontier based exploration

In this section, we will use a package that will generate the different navigation goals we had to select in the previous section in an autonomous way.

One simple yet powerful approach for autonomously exploring an area was proposed by B. Yamaouchi in his famous paper "[A Frontier-Based Approach for Autonomous Navigation](https://www.cs.cmu.edu/~motionplanning/papers/sbp_papers/integrated1/yamauchi_frontiers.pdf)" (1997). Quoting from him:

* "The central question in exploration is: Given what you know about the world, where should you move to gain as much new information as possible?"
* "The central idea behind frontier-based exploration is: To gain the most new information about the world, move to the boundary between open space and uncharted territory (i.e. frontiers)".
* "Frontiers are region on the boundary between open space and unexplored space"
* " Once frontiers have been detected within a particular evidence grid, the robot attempts to navigate to the nearest accessible, unvisited frontier."

Basically, it analyzes the current map searching for boundaries between unkwown space and free space. This boundaries are stored as frontiers, and will be candidate destinations for the ground robot. Then, the main procedure is to select as new goal the closest frontier point. In this way, the robot will be commanded

Another interesting paper: "[Frontier-based exploration for Autonomous Robots](https://arxiv.org/pdf/1806.03581)" by A. Topiwala et al.

#### Adding the autonomous exploration tool.

First, you have to download the ROS2 explore lite package in your ROS2 workspace. You can do it this way:

```
    (rssa_docker) > cd $HOME/ros2_ws/src/
    (rssa_docker) > git clone https://github.com/robo-friends/m-explore-ros2

```

Then, compile the workspace with colcon:

```
    (rssa_docker) > cd $HOME/ros2_ws
    (rssa_docker) > colcon build --symlink-install
```


Once the repository is compiled, you should just perform the SLAM as seen in the SLAM section and then, once the simulation is ready you should launch the ROS2 explore node as follows:

`(rssa_docker) > ros2 launch explore_lite explore.launch.py `

__Exercise_3__ 

Try to run the autonomous SLAM in different scenarios.


### Final remarks

With this part we complete the autonomous systems SLAM toolbox for the subject. 

For further SLAM exploration, you can go to the documentation of the ROS2 Slam Toolbox, which has several tutorials to generate a custom SLAM node.

* Slam Toolbox: a collection of SLAM packages and tools to easily develop a custom SLAM algorithm (https://docs.ros.org/en/jazzy/p/slam_toolbox/)

For more information, you can dig in the following manuscripts, which explain the foundations on Robotics.

* S. Thrun et al. "[Probabilistic Robotics](https://fama.us.es/discovery/fulldisplay?docid=alma991006678829704987&context=L&vid=34CBUA_US:VU1&tab=all_data_not_idus&lang=es)"

* Cartographer's Paper. W. Hess, D. Kohler, H. Rapp, and D. Andor, Real-Time Loop Closure in 2D LIDAR SLAM, in Robotics and Automation (ICRA), 2016 IEEE International Conference on. IEEE, 2016. pp. 1271–1278. 

* Slam handbook. A recent open-source book that summarizes the state-of-the-art on SLAM algorithms. Highly recommended!! (https://asrl.utias.utoronto.ca/~tdb/slam/slamhandbook.pdf)